# Asystent Pielegnacji Roslin - notebook demonstracyjny

Ten notebook pokazuje **krok po kroku**, jak dziala caly system. Kazdy element mozna uruchomic osobno, dzieki czemu latwo zrozumiec poszczegolne czesci:

1. **CLIP** - rozpoznawanie rosliny ze zdjecia.
2. **Wyszukiwarka (SerpAPI)** - pobieranie informacji o pielegnacji z internetu.
3. **RAG** - baza wektorowa: chunking, embeddingi, retrieval.
4. **Agent AI** - polaczenie wszystkiego: model sam decyduje, jakich narzedzi uzyc.

> Uwaga: czesc krokow wymaga kluczy API (OpenAI, SerpAPI). Ustaw je w pliku `.env` (skopiuj `.env.example`). Kroki bez kluczy (CLIP, baza wektorowa na recznych danych) zadzialaja od razu.

## 0. Przygotowanie

Instalacja zaleznosci (jednorazowo) oraz dodanie katalogu projektu do sciezki, aby moc importowac pakiet `src`.

In [ ]:
# Odkomentuj, jesli uruchamiasz po raz pierwszy:
# %pip install -r ../requirements.txt

import sys, os
sys.path.append(os.path.abspath('..'))  # aby zadzialal 'import src...'
print('Gotowe.')

## 1. CLIP - rozpoznawanie rosliny ze zdjecia

CLIP porownuje obraz z lista opisow tekstowych (np. *"a photo of a monstera houseplant"*) i wybiera najbardziej pasujacy. To klasyfikacja **zero-shot** - bez uczenia na wlasnych zdjeciach.

In [ ]:
from PIL import Image
from src.clip_classifier import get_classifier

# Podmien sciezke na swoje zdjecie rosliny:
image_path = '../data/sample_plant.jpg'

if os.path.exists(image_path):
    image = Image.open(image_path)
    classifier = get_classifier()
    predictions = classifier.classify(image, top_k=5)
    for p in predictions:
        print(f'{p.label:45s}  {p.confidence*100:5.1f}%')
else:
    print('Dodaj zdjecie do katalogu data/ i ustaw image_path.')

## 2. Wyszukiwanie informacji w internecie (SerpAPI)

Tworzymy zapytanie typu *"how to take care of monstera plant"*, pobieramy top wyniki Google i zamieniamy je na dokumenty tekstowe. Wymaga `SERPAPI_API_KEY`.

In [ ]:
from src.web_search import search_plant_care

documents = search_plant_care('monstera', num_results=5)
print(f'Pobrano {len(documents)} dokumentow.\n')
for d in documents:
    print('-', d.title)
    print('  ', d.url)
    print('  ', d.text[:160].replace('\n', ' '), '...')
    print()

## 3. RAG - baza wektorowa, embeddingi, retrieval

Dokumenty tniemy na fragmenty (chunking), zamieniamy na wektory (embeddingi) i wrzucamy do bazy wektorowej (FAISS). Potem zadajemy pytanie i pobieramy najtrafniejsze fragmenty.

Ponizej pokaz dziala nawet bez internetu - mozemy podac wlasne, recznie napisane "dokumenty".

In [ ]:
from src.rag import RAGPipeline
from src.web_search import SearchDocument

# Przykladowe dane (gdyby nie bylo internetu). Normalnie pochodza z SerpAPI.
demo_docs = [
    SearchDocument(
        title='Monstera care basics',
        url='https://example.com/monstera',
        text=('Monstera deliciosa likes bright, indirect light. Water when the top '
              '2-3 cm of soil is dry, usually once a week. Avoid overwatering, which '
              'causes yellow leaves and root rot. Fertilize monthly in spring and summer. '
              'It enjoys humidity and temperatures of 18-27 C.'),
    ),
]

rag = RAGPipeline()
n = rag.build_index(demo_docs, plant_name='monstera')
print(f'Zbudowano baze wektorowa z {n} fragmentow.\n')

context = rag.build_context('How often should I water it?')
print('Najtrafniejszy kontekst dla pytania o podlewanie:\n')
print(context)

## 4. Agent AI - wszystko razem

Agent dostaje zdjecie i pytanie uzytkownika. **Sam** decyduje, czy uzyc CLIP, wyszukiwarki i RAG, a nastepnie formuluje odpowiedz. Wymaga `OPENAI_API_KEY` (a do swiezych danych takze `SERPAPI_API_KEY`).

In [ ]:
from src.agent import PlantCareAgent
from PIL import Image

agent = PlantCareAgent()

# Jesli masz zdjecie - wczytaj je, aby agent mogl uzyc narzedzia CLIP:
if os.path.exists('../data/sample_plant.jpg'):
    agent.set_image(Image.open('../data/sample_plant.jpg'))

answer = agent.chat('Co to za roslina i jak o nia dbac?')
print(answer)
print('\nUzyte narzedzia:', agent.state.last_tools_used)

In [ ]:
# Rozmowa jest pamietana - mozna zadawac kolejne pytania:
print(agent.chat('A jak czesto ja nawozic?'))

## Podsumowanie

W tym notebooku pokazalismy wszystkie komponenty systemu osobno oraz ich polaczenie w agencie AI. Pelny interfejs graficzny uruchamiamy poleceniem:

```bash
streamlit run app.py
```